# Lead Data Exploration
Raw xlsx → anomaly audit → cleaned DataFrame

In [1]:
import pandas as pd

XLSX = "Restaurant Phone Numbers - Maple Take Home Round 2.xlsx"

raw = pd.read_excel(XLSX)
print(f"Shape: {raw.shape}")
raw.head()

Shape: (2355, 14)


,first_name,last_name,organization_website_url,Unnamed: 3,email,organization_street_address,organization_raw_address,organization_state,organization_city,organization_country,organization_postal_code,Google Reviews Count,Location Phone,Google Maps Url
0,Kyle,Svilar,http://www.313franklin.com,NaN,kyle.svilar@partners.mcd.com,NaN,"South Hill, Virginia 23970, US",Virginia,South Hill,United States,23970.0,558.0,1.434585e+10,https://www.google.com/maps/place/?q=place_id:...
1,Joseph,Galvin,http://www.galvinsonmain.com,NaN,NaN,155 W Main St,"155 W Main Street, Georgetown, KY 40324, US",Kentucky,Georgetown,United States,NaN,1678.0,1.502863e+10,https://www.google.com/maps/place/?q=place_id:...
2,John,Pepper,http://www.boloco.com,NaN,pepper@211.vc,1080 Boylston St,"1080 Boylston St, Boston, Massachusetts 02215, US",Massachusetts,Boston,United States,NaN,288.0,1.617431e+10,https://www.google.com/maps/place/?q=place_id:...
3,Jim,Hartman,http://www.jimmyhulas.com,NaN,NaN,2522 Aloma Ave,"2522 Aloma Avenue , Winter Park, Florida, USA,...",Florida,Winter Park,United States,NaN,1632.0,1.407791e+10,https://www.google.com/maps/place/?q=place_id:...
4,Zach,Hartman,http://www.jimmyhulas.com,NaN,zach@jimmyhulas.com,2522 Aloma Ave,"2522 Aloma Avenue , Winter Park, Florida, USA,...",Florida,Winter Park,United States,NaN,1632.0,1.407791e+10,https://www.google.com/maps/place/?q=place_id:...


## 1. `Unnamed: 3` — entirely null, drop it

In [3]:
print("Unnamed:3 null count:", raw["Unnamed: 3"].isnull().sum(), "/", len(raw))
df = raw.drop(columns=["Unnamed: 3"])
print(f"Columns after drop: {df.shape[1]}")

Unnamed:3 null count: 2355 / 2355
Columns after drop: 13


In [9]:
df

,first_name,last_name,organization_website_url,email,organization_street_address,organization_raw_address,organization_state,organization_city,organization_country,organization_postal_code,Google Reviews Count,Location Phone,Google Maps Url
0,Kyle,Svilar,http://www.313franklin.com,kyle.svilar@partners.mcd.com,NaN,"South Hill, Virginia 23970, US",Virginia,South Hill,United States,23970.0,558.0,1.434585e+10,https://www.google.com/maps/place/?q=place_id:...
1,Joseph,Galvin,http://www.galvinsonmain.com,NaN,155 W Main St,"155 W Main Street, Georgetown, KY 40324, US",Kentucky,Georgetown,United States,NaN,1678.0,1.502863e+10,https://www.google.com/maps/place/?q=place_id:...
2,John,Pepper,http://www.boloco.com,pepper@211.vc,1080 Boylston St,"1080 Boylston St, Boston, Massachusetts 02215, US",Massachusetts,Boston,United States,NaN,288.0,1.617431e+10,https://www.google.com/maps/place/?q=place_id:...
3,Jim,Hartman,http://www.jimmyhulas.com,NaN,2522 Aloma Ave,"2522 Aloma Avenue , Winter Park, Florida, USA,...",Florida,Winter Park,United States,NaN,1632.0,1.407791e+10,https://www.google.com/maps/place/?q=place_id:...
4,Zach,Hartman,http://www.jimmyhulas.com,zach@jimmyhulas.com,2522 Aloma Ave,"2522 Aloma Avenue , Winter Park, Florida, USA,...",Florida,Winter Park,United States,NaN,1632.0,1.407791e+10,https://www.google.com/maps/place/?q=place_id:...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2350,Louisa,Magoon,http://www.thegrandtearoom.com,louisa.magoon@thegrandtearoom.com,NaN,"Escondido, California",California,Escondido,United States,0.0,158.0,1.760234e+10,https://www.google.com/maps/place/?q=place_id:...
2351,Peter,Engert,http://www.kinokorestaurant.com,NaN,1007 Kane Concourse,"1007 Kane Concourse, Bay Harbor Islands, Flori...",Florida,Bay Harbor Islands,United States,NaN,155.0,1.305398e+10,https://www.google.com/maps/place/?q=place_id:...
2352,Aaron,Konieczny,http://www.goldenchickenhc.com,NaN,10605 W Forest Home Ave,"10605 west forest home avenue, hales corners, ...",Wisconsin,Hales Corners,United States,53130.0,177.0,1.414428e+10,https://www.google.com/maps/place/?q=place_id:...
2353,Amjad,Manna,http://www.manna-grill.com,manna@manna-grill.com,2913 Lincoln Ave,"2913 lincoln avenue, evansville, indiana, unit...",Indiana,Evansville,United States,NaN,756.0,1.812474e+10,https://www.google.com/maps/place/?q=place_id:...


## 2. Missing phones — rows we can never call, drop them

In [6]:
missing_phone = df[df["Location Phone"].isnull()]
print(f"Rows with no phone: {len(missing_phone)}")
missing_phone[["first_name", "last_name", "organization_city", "organization_state"]].head()

Rows with no phone: 40


,first_name,last_name,organization_city,organization_state
43,Chris,Galloway,Sunrise Beach,Missouri
154,George,Barriere,Medley,Florida
217,Dominick,Smith,Homewood,California
231,Brent,Schepper,Flagstaff,Arizona
267,Jonathan,Buford,Gilbert,Arizona


In [7]:
df = df[df["Location Phone"].notna()].copy()
print(f"Rows after dropping no-phone: {len(df)}")

Rows after dropping no-phone: 2315


## 3. Duplicate phones, same restaurant, multiple CRM contacts

Keep the first occurrence (lowest source row index), drop the rest.

In [8]:
phone_counts = df["Location Phone"].value_counts()
dups = phone_counts[phone_counts > 1]
print(f"Phone numbers with duplicates: {len(dups)}")
print(f"Extra rows that will be dropped: {dups.sum() - len(dups)}")
print()
print("Top duplicates:")
dups.head(10)

Phone numbers with duplicates: 217
Extra rows that will be dropped: 257

Top duplicates:


Location Phone
1.205982e+10    4
1.859371e+10    4
1.314933e+10    4
1.800767e+10    4
1.404390e+10    4
1.434529e+10    4
1.404881e+10    3
1.360256e+10    3
1.734668e+10    3
1.410997e+10    3
Name: count, dtype: int64

In [11]:
# Show a concrete example — same restaurant, 4 different contacts
example_phone = dups.index[0]
df[df["Location Phone"] == example_phone][
    [
        "first_name",
        "last_name",
        "organization_city",
        "organization_state",
        "organization_website_url",
    ]
]

,first_name,last_name,organization_city,organization_state,organization_website_url
769,Jessica,Morales,Birmingham,Alabama,http://www.ourbellinis.com
776,Melisa,Steele,Birmingham,Alabama,http://www.ourbellinis.com
782,Frances,Olliff,Birmingham,Alabama,http://www.ourbellinis.com
784,Doug,Hovanec,Birmingham,Alabama,http://www.ourbellinis.com


In [13]:
df = df.drop_duplicates(subset=["Location Phone"], keep="first").copy()
print(f"Rows after dedup: {len(df)}")

Rows after dedup: 2058


## 4. Zip code leading-zero truncation

Excel stores postal codes as `float64`. `07030` becomes `7030.0`, and any zip starting with `0` becomes `0` after int conversion. Zero-pad to 5 digits to recover them.

In [8]:
print("Postal code dtype:", df["organization_postal_code"].dtype)
print()

# Rows where raw value == 0.0  (leading zero was stripped by Excel)
zeros = df[df["organization_postal_code"] == 0]
print(f"Rows with postal_code == 0: {len(zeros)}")

# States where leading zeros are expected (New England + NJ + NY)
leading_zero_states = [
    "New Jersey",
    "Connecticut",
    "Massachusetts",
    "New Hampshire",
    "Vermont",
    "Rhode Island",
    "Maine",
    "New York",
]
affected = df[
    df["organization_state"].isin(leading_zero_states)
    & (df["organization_postal_code"].isnull() | (df["organization_postal_code"] == 0))
]
print(f"Rows in leading-zero states with broken zip: {len(affected)}")
affected[["organization_state", "organization_city", "organization_postal_code"]].head(10)

Postal code dtype: float64

Rows with postal_code == 0: 100
Rows in leading-zero states with broken zip: 185


,organization_state,organization_city,organization_postal_code
2,Massachusetts,Boston,NaN
18,Vermont,South Burlington,NaN
29,New York,New York,NaN
32,New Jersey,Belmar,NaN
35,Maine,Portland,NaN
53,Connecticut,South Windsor,NaN
74,Massachusetts,Cambridge,NaN
111,Connecticut,Stonington,NaN
195,Massachusetts,Brockton,NaN
214,Massachusetts,Boston,NaN


In [14]:
# Fix: zero-pad to 5 digits — "0" → "00000", "7030" → "07030"
def normalize_zip(raw):
    import math

    if raw is None or (isinstance(raw, float) and math.isnan(raw)):
        return None
    return str(int(raw)).zfill(5)


df["postal_code_normalized"] = df["organization_postal_code"].apply(normalize_zip)

# Show the fix in action for NJ rows
nj = df[df["organization_state"] == "New Jersey"][
    ["organization_postal_code", "postal_code_normalized"]
]
print(f"NJ rows: {len(nj)}")
nj.head(10)

NJ rows: 62


,organization_postal_code,postal_code_normalized
32,NaN,None
52,7604.0,07604
323,7834.0,07834
576,7960.0,07960
639,NaN,None
746,7960.0,07960
763,7601.0,07601
764,NaN,None
780,NaN,None
804,7401.0,07401


In [15]:
# Length distribution after fix
lengths = df["postal_code_normalized"].dropna().str.len().value_counts().sort_index()
print("Normalized zip length distribution:")
lengths

Normalized zip length distribution:


postal_code_normalized
5    1081
Name: count, dtype: int64

## 5. Final cleaned shape + null summary

In [16]:
print(f"Raw rows:     {len(raw)}")
print(f"Cleaned rows: {len(df)}")
print(f"Dropped:      {len(raw) - len(df)}")
print()
print("Null counts in cleaned df:")
df.isnull().sum().sort_values(ascending=False)

Raw rows:     2355
Cleaned rows: 2058
Dropped:      297

Null counts in cleaned df:


organization_postal_code       977
postal_code_normalized         977
email                          768
organization_street_address    135
organization_city                9
Google Reviews Count             8
organization_state               3
last_name                        1
first_name                       0
organization_website_url         0
organization_raw_address         0
organization_country             0
Location Phone                   0
Google Maps Url                  0
dtype: int64

## 6. Timezone inference coverage

How many leads will be callable (have a timezone) vs silently skipped?

In [17]:
import math
import sys

sys.path.insert(0, "src")

from mystery_shop.ingest.normalize import _infer_timezone


def _safe_state(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    return str(v)


df["timezone"] = df.apply(
    lambda r: _infer_timezone(
        r["postal_code_normalized"], _safe_state(r.get("organization_state"))
    ),
    axis=1,
)

tz_counts = df["timezone"].value_counts(dropna=False)
null_tz = df["timezone"].isnull().sum()
print(f"Leads with timezone:    {len(df) - null_tz} ({(len(df) - null_tz) / len(df) * 100:.1f}%)")
print(
    f"Leads without timezone: {null_tz} ({null_tz / len(df) * 100:.1f}%) — will be skipped by scheduler"
)
print()
print("Top 10 timezones:")
tz_counts.head(10)

ModuleNotFoundError: No module named 'pgeocode'

## 7. Google Reviews distribution

In [21]:
reviews = df["Google Reviews Count"].dropna()
print(f"Min: {reviews.min():.0f}  |  Median: {reviews.median():.0f}  |  Max: {reviews.max():.0f}")
print()
bins = [0, 10, 50, 100, 500, 1000, 99999]
labels = ["1-10", "11-50", "51-100", "101-500", "501-1000", "1000+"]
pd.cut(reviews, bins=bins, labels=labels).value_counts().sort_index()

Min: 0  |  Median: 796  |  Max: 24960



Google Reviews Count
1-10         82
11-50        61
51-100       39
101-500     475
501-1000    560
1000+       828
Name: count, dtype: int64